# 213 — Chunk Documents

Chunks each PDF into 200-400 token segments anchored to document sections, for use in vector search.

| Output | Description |
|---|---|
| `chunks.csv` | All chunks across all documents |

Token estimate: 1 token ≈ 4 characters. Target chunk size: ~1 200 characters (≈300 tokens).

Re-runnable: yes.

In [1]:
import sys, os, yaml
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv

project_root = Path(os.getcwd()).parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

load_dotenv(project_root / '.env')

from src.document.pdf_utils import extract_full_text
from src.document.config    import load_document_config

LAYER2_DIR       = project_root / 'data' / 'layer_2'
INTERMEDIATE_DIR = LAYER2_DIR / 'intermediate'
PDF_DIR          = LAYER2_DIR / 'regulatory_documents'
CONFIG_PATH      = LAYER2_DIR / 'document_config.yaml'

TARGET_CHARS = 1200   # ~300 tokens
MIN_CHARS    =  600   # ~150 tokens
MAX_CHARS    = 2000   # ~500 tokens

config = load_document_config(CONFIG_PATH)
docs   = config['documents']

print(f'Intermediate dir: {INTERMEDIATE_DIR}')
print(f'Documents       : {len(docs)}')


Intermediate dir: /Users/emillaurencepastor/Documents/github/graphrag-finserv-compliance/data/layer_2/intermediate
Documents       : 3


## Step 1 — Load sections

In [2]:
sections_df = pd.read_csv(LAYER2_DIR / 'sections.csv')
print(f'Sections loaded: {len(sections_df)}')
print(sections_df[['regulation_id', 'section_id', 'section_type']].head(10).to_string())


Sections loaded: 106
  regulation_id   section_id section_type
0       APG-223   APG-223-S1         core
1       APG-223   APG-223-S2         core
2       APG-223   APG-223-S3         core
3       APG-223   APG-223-S4         core
4       APG-223   APG-223-S5         core
5       APG-223   APG-223-S6         core
6       APG-223   APG-223-S7         core
7       APG-223   APG-223-S8         core
8       APG-223   APG-223-S9         core
9       APG-223  APG-223-S10   attachment


## Step 2 — Chunking functions

In [3]:
# extract_full_text() is imported from src.document.pdf_utils

def chunk_text(text: str) -> list:
    """
    Split text into chunks of approximately TARGET_CHARS characters.
    Respects paragraph boundaries (double newlines) first,
    then single newlines, then hard-splits at MAX_CHARS.
    Returns list of non-empty strings.
    """
    if not text or not text.strip():
        return []

    paragraphs = [p.strip() for p in text.split('\n\n') if p.strip()]
    chunks  = []
    current = ''

    for para in paragraphs:
        # Hard-split a single paragraph that exceeds MAX_CHARS
        if len(para) > MAX_CHARS:
            sub_lines = [ln.strip() for ln in para.splitlines() if ln.strip()]
            for line in sub_lines:
                if len(current) + len(line) + 1 <= MAX_CHARS:
                    current = (current + '\n' + line).strip()
                else:
                    if len(current) >= MIN_CHARS:
                        chunks.append(current)
                    current = line
            continue

        joined = (current + '\n\n' + para).strip() if current else para
        if len(joined) > MAX_CHARS and len(current) >= MIN_CHARS:
            chunks.append(current)
            current = para
        else:
            current = joined

    if current.strip():
        chunks.append(current.strip())

    return chunks


def assign_section(chunk_text_content: str, section_lookup: list, default_sid: str) -> str:
    """
    Assign the best matching section_id to a chunk by looking for
    section titles appearing in the chunk text.
    Falls back to default_sid if no title matches.
    """
    lower = chunk_text_content.lower()
    for sid, title_lower in section_lookup:
        if title_lower and len(title_lower) > 4 and title_lower in lower:
            return sid
    return default_sid


## Step 3 — Process all documents

In [4]:
all_chunks = []

for doc in docs:
    rid      = doc['regulation_id']
    pdf_path = PDF_DIR / doc['pdf_path']

    doc_sections = (
        sections_df[sections_df['regulation_id'] == rid].copy()
        if 'regulation_id' in sections_df.columns
        else sections_df.copy()
    )

    print(f'\n{rid}: {len(doc_sections)} sections')

    if not pdf_path.exists():
        raise FileNotFoundError(f'PDF not found: {pdf_path}')

    full_text = extract_full_text(pdf_path)
    print(f'  PDF characters: {len(full_text):,}')

    raw_chunks = chunk_text(full_text)
    print(f'  Raw chunks: {len(raw_chunks)}')

    # Build section title lookup for assignment heuristic
    default_sid    = doc_sections.iloc[0]['section_id'] if not doc_sections.empty else f'{rid}-S1'
    section_lookup = [
        (row['section_id'], str(row.get('title', '')).lower())
        for _, row in doc_sections.iterrows()
    ]

    chunk_records = []
    for i, chunk_content in enumerate(raw_chunks):
        sid = assign_section(chunk_content, section_lookup, default_sid)
        chunk_records.append({
            'chunk_id':        f'{rid}-CHUNK-{i + 1:04d}',
            'section_id':      sid,
            'text':            chunk_content,
            'token_count':     len(chunk_content) // 4,
            'chunk_index':     i,
            'source_document': rid,
        })

    all_chunks.extend(chunk_records)
    print(f'  Chunks created: {len(chunk_records)}')
    avg_tokens = sum(c['token_count'] for c in chunk_records) / max(len(chunk_records), 1)
    print(f'  Avg token estimate: {avg_tokens:.0f}')



APS-112: 68 sections
  PDF characters: 119,106
  Raw chunks: 65
  Chunks created: 65
  Avg token estimate: 457

APG-223: 10 sections
  PDF characters: 71,539
  Raw chunks: 37
  Chunks created: 37
  Avg token estimate: 483

APS-220: 28 sections
  PDF characters: 70,877
  Raw chunks: 36
  Chunks created: 36
  Avg token estimate: 491


## Step 4 — Write and validate

In [5]:
chunks_df = pd.DataFrame(all_chunks)
chunks_df.to_csv(LAYER2_DIR / 'chunks.csv', index=False)
print(f'Written chunks.csv to data/layer_2/: {len(chunks_df)} total chunks')

# Validation
all_section_ids = set(sections_df['section_id'].tolist())

unknown = chunks_df[~chunks_df['section_id'].isin(all_section_ids)]
if not unknown.empty:
    print(f'\n\u26a0  {len(unknown)} chunks reference unknown section_ids')
else:
    print('\n\u2713 All chunks reference valid section_ids')

sections_with_chunks    = set(chunks_df['section_id'].unique())
sections_without_chunks = all_section_ids - sections_with_chunks
print(f'\u26a0  Sections without chunks: {len(sections_without_chunks)}')
if sections_without_chunks:
    for sid in sorted(sections_without_chunks):
        print(f'    {sid}')

print('\nSummary per document:')
print(
    chunks_df.groupby('source_document').agg(
        chunks=('chunk_id', 'count'),
        avg_tokens=('token_count', 'mean'),
        total_tokens=('token_count', 'sum'),
    ).round(0).to_string()
)


Written chunks.csv to data/layer_2/: 138 total chunks

✓ All chunks reference valid section_ids
⚠  Sections without chunks: 61
    APG-223-S10
    APG-223-S9
    APS-112-ATT-A-LMI
    APS-112-ATT-A-LVR-CALC
    APS-112-ATT-A-NONSTD-RES-RW
    APS-112-ATT-A-RES-PROP
    APS-112-ATT-A-TABLE1
    APS-112-ATT-A-TABLE2
    APS-112-ATT-A-TABLE3
    APS-112-ATT-A-TABLE4
    APS-112-ATT-B-COVERED-BONDS
    APS-112-ATT-B-INTRO
    APS-112-ATT-B-PSE
    APS-112-ATT-B-S17
    APS-112-ATT-B-S18
    APS-112-ATT-B-S22-24
    APS-112-ATT-B-S25
    APS-112-ATT-B-S30-31
    APS-112-ATT-B-S32
    APS-112-ATT-B-S33
    APS-112-ATT-B-S39-40
    APS-112-ATT-B-S41
    APS-112-ATT-B-S42
    APS-112-ATT-B-S43-44
    APS-112-ATT-B-TABLE5
    APS-112-ATT-B-TABLE6
    APS-112-ATT-B-TABLE7
    APS-112-ATT-B-TABLE8
    APS-112-ATT-B-TABLE9
    APS-112-ATT-D-P31
    APS-112-ATT-E
    APS-112-ATT-E-TBL19
    APS-112-ATT-E-TBL20
    APS-112-ATT-F
    APS-112-ATT-F-TBL22
    APS-112-ATT-G
    APS-112-ATT-G-COLL-MATURI